# E：T4 首轮核验与答案修复

在 Colab 选择 **代码执行程序 → 更改运行时类型 → T4 GPU**，依次执行单元格。
Kaggle 可导入同一 notebook，启用 GPU 和 Internet，并把实验包添加到私有输入。

上传 `answer_repair_v2_gpu.tar.gz`；它包含冻结代码和 FEVER/NQ 各 50 条开发样本。
先运行 FEVER 10 条工程 smoke，检查全部输出，再运行可选开发实验。
开发样本排除了 Diagnosis v1/v2 的验证、测试 ID。Smoke 是开发集的子集。
参考答案仅用于离线评分，不会传入修复模型。引用合法不代表语义支持已验证。


In [ ]:
import json, os, sys, subprocess, hashlib, tarfile, shutil
from pathlib import Path
import torch

assert sys.version_info >= (3, 11), "需要 Python 3.11+"
assert torch.cuda.is_available(), "请先选择 GPU 运行时；当前 PyTorch 看不到 CUDA"
print(json.dumps({"python": sys.version, "torch": torch.__version__,
                  "cuda": torch.version.cuda,
                  "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}, indent=2))
WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/content')
OUTPUT = WORK / 'answer_repair_v2_outputs'
OUTPUT.mkdir(parents=True, exist_ok=True)


In [ ]:
# 已从 Colab 左侧文件面板上传时直接复用；否则弹出上传框。
EXPECTED_BUNDLE_SHA256 = '2f4a3d2a035ae6c59449a54efc820d2f7b22046b8071f6bf05d81322e58d4aef'
archive = WORK / 'answer_repair_v2_gpu.tar.gz'
if not archive.exists() and Path('/kaggle/input').exists():
    candidates = list(Path('/kaggle/input').rglob('answer_repair_v2_gpu.tar.gz'))
    assert len(candidates) == 1, '请在 Kaggle 添加唯一的实验包输入'
    archive = candidates[0]
elif not archive.exists():
    from google.colab import files
    uploaded = files.upload()
    assert 'answer_repair_v2_gpu.tar.gz' in uploaded, '请选择 answer_repair_v2_gpu.tar.gz'
    archive.write_bytes(uploaded['answer_repair_v2_gpu.tar.gz'])

assert hashlib.sha256(archive.read_bytes()).hexdigest() == EXPECTED_BUNDLE_SHA256, '实验包哈希不一致'
extract_root = WORK / ('e_bundle_' + EXPECTED_BUNDLE_SHA256[:12])
BUNDLE = extract_root / 'answer_repair_v2_gpu'
if not BUNDLE.exists():
    extract_root.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive) as handle:
        handle.extractall(extract_root, filter='data')
checksums = json.loads((BUNDLE / 'bundle_checksums.json').read_text())
assert all(hashlib.sha256((BUNDLE / name).read_bytes()).hexdigest() == value
           for name, value in checksums.items()), '代码或数据文件校验失败'
manifest = json.loads((BUNDLE / 'manifest.json').read_text())
print('包校验通过:', len(checksums), '个文件')
print(json.dumps({ds: {'development': item['development'], 'smoke': item['smoke']}
                  for ds, item in manifest['datasets'].items()}, indent=2))


In [ ]:
def run_logged(argv, log_name, cwd=BUNDLE):
    # 显式转发子进程输出，避免 notebook 只显示“正在执行”。
    env = {**os.environ, 'PYTHONUNBUFFERED': '1'}
    with (OUTPUT / log_name).open('a', encoding='utf-8') as log:
        log.write('COMMAND ' + json.dumps(argv) + '\n')
        log.flush()
        process = subprocess.Popen(argv, cwd=cwd, env=env, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
            returncode = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
    if returncode:
        raise subprocess.CalledProcessError(returncode, argv)

# 使用宿主 CUDA PyTorch；环境检查记录实际安装后的版本。
run_logged([sys.executable, '-m', 'pip', 'install', '-r',
            str(BUNDLE / 'requirements-answer-repair.txt')], 'install.log')
run_logged([sys.executable, 'scripts/check_answer_repair_environment.py',
            '--require-cuda', '--out', str(OUTPUT / 'environment.json')], 'environment.log')
run_logged([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests'], 'tests.log')
shutil.copy2(BUNDLE / 'manifest.json', OUTPUT / 'manifest.json')
shutil.copy2(BUNDLE / 'bundle_checksums.json', OUTPUT / 'bundle_checksums.json')


In [ ]:
SMOKE_DATASET = 'fever'
SMOKE_RESULT = OUTPUT / f'{SMOKE_DATASET}_smoke10_verify.jsonl'
SMOKE_ARGS = [sys.executable, '-u', 'scripts/run_answer_repair.py', '--dataset', SMOKE_DATASET,
              '--c1', str(BUNDLE / f'inputs/{SMOKE_DATASET}_smoke10_c1.jsonl'),
              '--all', '--out', str(SMOKE_RESULT)]
run_logged(SMOKE_ARGS + ['--check-only'], 'smoke_preflight.log')


In [ ]:
# 真实 7B NF4 调用；首次需要下载模型。每条追加保存，可在同一环境断点续跑。
run_logged(SMOKE_ARGS, 'smoke_run.log')


In [ ]:
summary = json.loads(SMOKE_RESULT.with_suffix('.summary.json').read_text())
print(json.dumps(summary, indent=2, ensure_ascii=False))
source_rows = {str(row['sample_id']): row for row in
               map(json.loads, (BUNDLE / f'inputs/{SMOKE_DATASET}_smoke10_c1.jsonl').read_text().splitlines())}
results = list(map(json.loads, SMOKE_RESULT.read_text().splitlines()))
review = []
for result in results:
    source = source_rows[result['sample_id']]
    row = {**result, 'reference_answer_or_label': source['gold_answer_or_label'],
           'input_evidence': source['evidence'],
           'manual_review': {'supports_final_answer': None, 'notes': ''}}
    review.append(row)
    print(json.dumps({k: result.get(k) for k in
                     ('sample_id', 'status', 'initial_answer', 'final_answer', 'citations', 'validation_error')},
                    ensure_ascii=False))
review_file = OUTPUT / f'{SMOKE_DATASET}_smoke10_review.jsonl'
review_file.write_text(''.join(json.dumps(row, ensure_ascii=False) + '\n' for row in review))
print('离线人工审核文件:', review_file)


## 可选开发实验

先审核 smoke 中的 JSON、引用、改对和改错。确认执行正常后，将下一格的
`RUN_DEVELOPMENT` 改为 `True`。它会运行 FEVER/NQ 各 50 条的 Verify 和 Reanswer
对照，总计最多 200 次额外生成。这是开发结果，不能当作最终独立测试结果。


In [ ]:
RUN_DEVELOPMENT = False
if RUN_DEVELOPMENT:
    for dataset in ('fever', 'nq'):
        for strategy in ('verify_repair', 'reanswer'):
            out = OUTPUT / f'{dataset}_dev50_{strategy}.jsonl'
            run_logged([sys.executable, '-u', 'scripts/run_answer_repair.py', '--dataset', dataset,
                        '--c1', str(BUNDLE / f'inputs/{dataset}_dev50_c1.jsonl'), '--all',
                        '--strategy', strategy, '--out', str(out)], f'{dataset}_{strategy}.log')
else:
    print('开发实验未启动；先审核 smoke。')


In [ ]:
# 导出已有结果和环境记录；模型权重不打包。
export = shutil.make_archive(str(WORK / 'answer_repair_v2_outputs'), 'zip', OUTPUT)
print('结果包:', export)
if not Path('/kaggle/working').exists():
    from google.colab import files
    files.download(export)
